In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

text = ["다음달 공휴일은 몇일 있는가?", "새 의자의 모델명과 가격을 검색", "오늘 오후 날씨 예상 좀 해줘"]

# 사전에 없는 단어
tokenizer = Tokenizer(oov_token='<UNK>')
tokenizer.fit_on_texts(text)

voca_size = len(tokenizer.word_counts) + 1

tokenizer.word_index, voca_size

({'<UNK>': 1,
  '다음달': 2,
  '공휴일은': 3,
  '몇일': 4,
  '있는가': 5,
  '새': 6,
  '의자의': 7,
  '모델명과': 8,
  '가격을': 9,
  '검색': 10,
  '오늘': 11,
  '오후': 12,
  '날씨': 13,
  '예상': 14,
  '좀': 15,
  '해줘': 16},
 16)

In [28]:
# 단어사전을 기반으로 문자를 숫자로 변경하기
input = ["오늘 뭐 먹을껀가요?", "날씨 참 좋다", "몇일 있는가 있는가 새"]
input_token = tokenizer.texts_to_sequences(input)
input_token

[[11, 1, 1], [13, 1, 1], [4, 5, 5, 6]]

In [30]:
# 입력한 문장에 따라서 길이가 다름
from tensorflow.keras.preprocessing.sequence import pad_sequences

input_pad = pad_sequences(input_token, maxlen=4, padding="post") # pre 또는 post
input_pad

array([[11,  1,  1,  0],
       [13,  1,  1,  0],
       [ 4,  5,  5,  6]], dtype=int32)

In [ ]:
# 단어 횟수
text1 = ["오늘 날씨 몇일", "의자 의자 날씨 예상"]
tokenizer.fit_on_texts(text1)
print(tokenizer.word_index)
tokenizer.texts_to_matrix(text1, mode="count") #  mode="count", binary=(0,1)

{'<UNK>': 1, '날씨': 2, '의자': 3, '몇일': 4, '오늘': 5, '예상': 6, '다음달': 7, '공휴일은': 8, '있는가': 9, '새': 10, '의자의': 11, '모델명과': 12, '가격을': 13, '검색': 14, '오후': 15, '좀': 16, '해줘': 17}


array([[0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 1., 2., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.]])

In [1]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/language.csv")
df

,text,label,label_name
0,I love programming in Java,0,Java
1,Python is great for data science,1,Python
2,JavaScript is used for web development,2,JavaScript
3,PHP is a server-side scripting language,3,PHP
4,Java is a popular language,0,Java
5,I write Python code everyday,1,Python
6,JavaScript is great for front-end development,2,JavaScript
7,PHP is used for backend development,3,PHP
8,I love writing Java code,0,Java
9,Python is amazing for machine learning,1,Python


In [2]:
df['label_name'].value_counts()

label_name
Java          4
Python        4
JavaScript    4
PHP           4
Name: count, dtype: int64

In [3]:
texts = df['text'].values
texts

array(['I love programming in Java', 'Python is great for data science',
       'JavaScript is used for web development',
       'PHP is a server-side scripting language',
       'Java is a popular language', 'I write Python code everyday',
       'JavaScript is great for front-end development',
       'PHP is used for backend development', 'I love writing Java code',
       'Python is amazing for machine learning',
       'JavaScript helps to build interactive websites',
       'PHP is used to create dynamic web pages',
       'Java has a strong community', 'Python is easy to learn',
       'JavaScript makes websites dynamic',
       'PHP is a server-side language for web development'], dtype=object)

In [118]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

voca_size = 1000

tokenizer = Tokenizer(num_words=voca_size, oov_token="<UNK>")
tokenizer.fit_on_texts(texts)

tokenizer.word_index

{'<UNK>': 1,
 'is': 2,
 'for': 3,
 'java': 4,
 'python': 5,
 'javascript': 6,
 'development': 7,
 'php': 8,
 'a': 9,
 'i': 10,
 'used': 11,
 'web': 12,
 'language': 13,
 'to': 14,
 'love': 15,
 'great': 16,
 'server': 17,
 'side': 18,
 'code': 19,
 'websites': 20,
 'dynamic': 21,
 'programming': 22,
 'in': 23,
 'data': 24,
 'science': 25,
 'scripting': 26,
 'popular': 27,
 'write': 28,
 'everyday': 29,
 'front': 30,
 'end': 31,
 'backend': 32,
 'writing': 33,
 'amazing': 34,
 'machine': 35,
 'learning': 36,
 'helps': 37,
 'build': 38,
 'interactive': 39,
 'create': 40,
 'pages': 41,
 'has': 42,
 'strong': 43,
 'community': 44,
 'easy': 45,
 'learn': 46,
 'makes': 47}

In [119]:
# 단어 빈도수 구하기
x = tokenizer.texts_to_matrix(texts, mode="count")
y = df[['label']].values
x.shape, y.shape

((16, 1000), (16, 1))

In [120]:
# 8:2로 나누기
from sklearn.model_selection import train_test_split

# stratify=y는 y클래스를 일정한 비율로 나누기
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123, stratify=y)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((12, 1000), (4, 1000), (12, 1), (4, 1))

In [121]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = keras.Sequential([
    Input(shape=(1000,)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(4, activation="softmax"),
])

model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 128)            │       128,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 136,644 (533.77 KB)

 Trainable params: 136,644 (533.77 KB)

 Non-trainable params: 0 (0.00 B)

In [122]:
model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [123]:
model.input_shape, model.input_dtype, x_train.shape

((None, 1000), 'float32', (12, 1000))

In [124]:
history = model.fit(
    x_train, 
    y_train, 
    validation_data=(x_test, y_test),
    epochs=40,
)

Epoch 1/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 694ms/step - accuracy: 0.1667 - loss: 1.3968 - val_accuracy: 0.2500 - val_loss: 1.3946
Epoch 2/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.2500 - loss: 1.3638 - val_accuracy: 0.2500 - val_loss: 1.3832
Epoch 3/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4167 - loss: 1.3330 - val_accuracy: 0.2500 - val_loss: 1.3723
Epoch 4/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.6667 - loss: 1.3036 - val_accuracy: 0.2500 - val_loss: 1.3612
Epoch 5/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8333 - loss: 1.2756 - val_accuracy: 0.2500 - val_loss: 1.3498
Epoch 6/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9167 - loss: 1.2485 - val_accuracy: 0.2500 - val_loss: 1.3387
Epoch 7/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.9167 - loss: 1.2224 - val_accuracy: 0.5000 - val_loss: 1.3277
Epoch 8/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.9167 - loss: 1.1967 - val_accuracy: 0.5000 - val_loss: 1.3168

In [133]:
import numpy as np
xs = model.predict(x_test)
for y, x in zip(y_test, xs):
    print(df['label_name'][y[0]], df['label_name'][np.argmax(x)])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Python Python
JavaScript JavaScript
PHP PHP
Java Java


In [147]:
sample = [
    "Modern web applications often rely on JavaScript for interactivity.",
    "Reliable enterprise systems are frequently built using Java.",
    "Automation tasks become easier when developed with Python.",
    "Many backend websites continue to be powered by PHP.",
    "Scalable business software commonly benefits from Java technologies.",
    "Machine learning projects are often implemented using Python.",
    "Developers use JavaScript across both frontend and backend environments.",
    "Database-driven applications can be rapidly created with PHP.",
    "Long-term maintainability remains a key strength of Java.",
    "Dynamic user experiences are commonly delivered through JavaScript.",
    "A rich ecosystem of libraries makes Python highly productive.",
    "Popular content management platforms are largely based on PHP.",
    "Large organizations frequently choose Java for critical applications.",
    "The extensive tooling around JavaScript accelerates development.",
    "Beginners often find Python easier to learn and use.",
    "Content-focused web solutions are regularly developed with PHP."
]

In [148]:
label = {0: "Java", 1: "Python", 2: "Javascript", 3:"PHP"}

In [150]:
sample_text = tokenizer.texts_to_matrix(sample, mode="count")
pred = model.predict(sample_text)
pred_labels = pred.argmax(axis=1)
for s, p in zip(sample, pred_labels):
    print(f"{s:80s} => {label[p]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Modern web applications often rely on JavaScript for interactivity.              => Javascript
Reliable enterprise systems are frequently built using Java.                     => Java
Automation tasks become easier when developed with Python.                       => Java
Many backend websites continue to be powered by PHP.                             => PHP
Scalable business software commonly benefits from Java technologies.             => Java
Machine learning projects are often implemented using Python.                    => PHP
Developers use JavaScript across both frontend and backend environments.         => Java
Database-driven applications can be rapidly created with PHP.                    => PHP
Long-term maintainability remains a key strength of Java.                        => Java
Dynamic user experiences are commonly delivered through JavaScript.              => Javascript
A rich ecosystem of libraries makes Python highly productive.  